# Trace Count v7: pure context-length extension of v2

这个 notebook 只改变 v2 的 prompt 长度，count 始终为 1–10：

- length 1024：non-thinking + thinking；
- length 2048：non-thinking + thinking。

Main 一共训练四个模型。trace index 与最终答案共享 `<1>...<10>` 数字 token。

评估同时记录 `tf_accuracy` 和 `ar_accuracy`。其中 `tf_accuracy` 是 teacher-forced answer position 上的 final-count readout；`ar_accuracy` 是从 prompt 后 autoregressively 生成到 final answer 后的准确率。报告里的 `accuracy` 默认使用 `ar_accuracy`。
            

## 1. Setup

In [1]:
from __future__ import annotations

from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
INSTALL_DEPS = False

if IN_COLAB:
    repo_dir = Path("/content/Synthetic_CoT_NiaH_Count")
    cwd = Path.cwd()
    if (cwd / ".git").exists() or (cwd / "README.md").exists():
        repo_dir = cwd
    elif not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40", "pandas", "matplotlib", "tqdm"], check=True)

import pandas as pd
from IPython.display import Markdown, display, Image

display(Markdown(f"**Repo root:** `{ROOT}`"))

**Repo root:** `/content/Synthetic_CoT_NiaH_Count`

## 2. Runtime settings

In [2]:
PRESET = "main"  # "debug" or "main"
OUT_ROOT = "runs/trace_count_v7_cot_advantage"
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
SKIP_COMPLETED = True
print({"PRESET": PRESET, "OUT_ROOT": OUT_ROOT, "DEVICE": DEVICE})
            

{'PRESET': 'main', 'OUT_ROOT': 'runs/trace_count_v7_cot_advantage', 'DEVICE': 'cuda'}


## 3. Run v7 sweep

In [ ]:
from synthetic_counting_extensions.v7_v8_sweeps import preset_configs, run_sweep

settings = preset_configs("v7", PRESET)
display(pd.DataFrame([vars(cfg) | {"effective_batch_size": cfg.effective_batch_size} for cfg in settings]))
display(Markdown(f"**Training runs:** `{len(settings)} settings × 2 models = {2 * len(settings)}`"))

combined = run_sweep("v7", PRESET, OUT_ROOT, skip_completed=SKIP_COMPLETED, device=DEVICE)
display(combined.head())
display(combined.groupby(["setting", "mode"], as_index=False)["accuracy"].mean())
            

,experiment,preset,seed,seq_len,train_count_min,train_count_max,eval_count_min,eval_count_max,noise_vocab_size,marker_vocab_size,...,warmup_steps,weight_decay,log_every,eval_examples_per_count,ar_examples_per_count,n_layer,n_head,n_embd,device,effective_batch_size
0,v7,main,1234,1024,1,10,1,10,64,10,...,500,0.1,50,200,50,4,4,256,cuda,128
1,v7,main,1234,2048,1,10,1,10,64,10,...,500,0.1,50,150,30,4,4,256,cuda,128


**Training runs:** `2 settings × 2 models = 4`

v7 nonthinking:   0%|          | 0/10000 [00:00<?, ?it/s]

eval nonthinking:   0%|          | 0/2000 [00:00<?, ?it/s]

v7 thinking:   0%|          | 0/10000 [00:00<?, ?it/s]

eval thinking:   0%|          | 0/2000 [00:00<?, ?it/s]

v7 nonthinking:   0%|          | 0/10000 [00:00<?, ?it/s]

## 4. Display reports

In [ ]:
for run in Path(OUT_ROOT).glob("v7_*"):
    report = run / "report" / "report.html"
    figures = [
        run / "figures" / "accuracy_by_count.png",
        run / "figures" / "accuracy_by_validation_split.png",
    ]
    if figures[0].exists():
        display(Markdown(f"## {run.name}\nReport: `{report}`"))
        for fig in figures:
            if fig.exists():
                display(Image(filename=str(fig)))
            

## 5. Save / GitHub / disconnect

In [ ]:
# Save result folder to Google Drive.
SAVE_TO_DRIVE = True
DRIVE_DEST_ROOT = "/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/Synthetic_CoT_NiaH_Count/colab_results"

if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    dest_root = Path(DRIVE_DEST_ROOT)
    dest_root.mkdir(parents=True, exist_ok=True)
    if "RUN_DIR" in globals() and RUN_DIR is not None:
        src = Path(RUN_DIR)
    elif "OUT_ROOT" in globals():
        src = Path(OUT_ROOT)
    else:
        src = None
    if src is not None and src.exists():
        dest = dest_root / src.name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(src, dest)
        display(Markdown(f"**Saved to Drive:** `{dest}`"))
    else:
        display(Markdown("No RUN_DIR/OUT_ROOT found to save."))
else:
    display(Markdown("Drive save skipped."))

In [ ]:
# Optional: commit and push notebook/code changes to GitHub.
PUSH_TO_GITHUB = False
GIT_COMMIT_MESSAGE = "Add synthetic counting experiment notebook"

if PUSH_TO_GITHUB:
    subprocess.run(["git", "status", "--short"], check=False)
    subprocess.run(["git", "add", "notebooks", "synthetic_counting_extensions", "scripts"], check=True)
    subprocess.run(["git", "commit", "-m", GIT_COMMIT_MESSAGE], check=True)
    subprocess.run(["git", "push"], check=True)
else:
    display(Markdown("GitHub push skipped. Set `PUSH_TO_GITHUB = True` after checking the diff."))

In [ ]:
# Optional: disconnect Colab runtime after saving.
AUTO_DISCONNECT_AFTER_SAVE = False

if AUTO_DISCONNECT_AFTER_SAVE and IN_COLAB:
    from google.colab import runtime
    runtime.unassign()
else:
    display(Markdown("Auto-disconnect skipped."))